# Huấn luyện CNN — Phân loại trạng thái mắt (nhắm / mở)

## Mục lục — Mục đích từng phần

| Phần | Mục đích |
|------|----------|
| **1–2** | Chuẩn bị dữ liệu: cân bằng lớp (Closed đầy đủ; Open lấy mẫu ngẫu nhiên từ 5 hướng nhìn), chia **80% train / 20% validation**, sao chép vào `/kaggle/working/smartdrive_data`; kiểm đếm đối chiếu. |
| **3** | *(Tuỳ chọn)* Thống kê số lượng ảnh và video theo thư mục trong dataset SDDD gốc. |
| **4–5** | Cấu hình siêu tham số, **`tf.data` + `image_dataset_from_directory`**, tăng cường bằng lớp Keras ở đầu mô hình, đầu vào **grayscale 64×64**. |
| **6–7** | Xây dựng kiến trúc **CNN**, compile mô hình phân loại nhị phân. |
| **8–9** | Huấn luyện: callback **ModelCheckpoint**, **EarlyStopping**, ước lượng thời gian theo epoch. |
| **10–11** | Đánh giá trên tập validation: độ chính xác, ma trận nhầm lẫn, báo cáo Precision/Recall/F1, **ROC-AUC**, kích thước file mô hình. |

---
**Trình tự tiếp theo:** ô markdown *Chuẩn bị và phân chia dữ liệu*, sau đó ô mã thực thi (sao chép và chia **80% / 20%**).

## Chuẩn bị và phân chia dữ liệu

**Mục đích:** Thu thập toàn bộ ảnh nhãn **Closed_Eyes**; lấy mẫu ngẫu nhiên **Open_Eyes** từ năm thư mục hướng nhìn (giới hạn mỗi thư mục); chia **80% train / 20% validation**; sao chép vào `/kaggle/working/smartdrive_data` và in tóm tắt số tệp.

In [2]:
import os
import shutil
import random
from tqdm import tqdm

# === Cấu hình (Kaggle) ===
EYE_GAZE_ROOT = (
    "/kaggle/input/datasets/rahimulmazumdar/simulated-driver-distraction-dataset-sddd/"
    "Dataset/Image Data/Eye Gaze"
)
SOURCE_CLOSED = os.path.join(EYE_GAZE_ROOT, "Closed")
# 5 hướng mắt mở: 100 ảnh / thư mục -> 500 ảnh Open (cân với ~500 Closed)
OPEN_SUBFOLDERS = ["Front", "Down", "Left", "Right", "Up"]
SAMPLES_PER_OPEN_FOLDER = 100
SPLIT_RATIO = 0.8
RANDOM_SEED = 42

BASE_DIR = "/kaggle/working/smartdrive_data"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_image_paths(folder: str) -> list:
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Không tìm thấy thư mục: {folder}")
    out = []
    for name in os.listdir(folder):
        p = os.path.join(folder, name)
        if os.path.isfile(p) and os.path.splitext(name)[1].lower() in IMAGE_EXTS:
            out.append(p)
    return sorted(out)


def copy_with_unique_name(src: str, dst_dir: str, prefix: str, counter: int) -> None:
    base = os.path.basename(src)
    stem, ext = os.path.splitext(base)
    dst_name = f"{prefix}_{counter:05d}_{stem}{ext}"
    dst = os.path.join(dst_dir, dst_name)
    shutil.copy2(src, dst)


def split_list_and_copy(paths: list, train_dir: str, val_dir: str, prefix: str) -> None:
    if not paths:
        print(f"[Cảnh báo] Danh sách đường dẫn rỗng — bỏ qua sao chép (prefix={prefix}).")
        return
    random.shuffle(paths)
    n = len(paths)
    cut = int(n * SPLIT_RATIO)
    train_paths, val_paths = paths[:cut], paths[cut:]
    for i, p in enumerate(tqdm(train_paths, desc=f"Sao chép tập huấn luyện [{prefix}]")):
        copy_with_unique_name(p, train_dir, prefix, i)
    for i, p in enumerate(tqdm(val_paths, desc=f"Sao chép tập kiểm định [{prefix}]")):
        copy_with_unique_name(p, val_dir, prefix, i)


random.seed(RANDOM_SEED)

# Xóa lần chạy cũ trong /kaggle/working (tránh trộn ảnh)
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)
for split in (TRAIN_DIR, VAL_DIR):
    os.makedirs(os.path.join(split, "Closed_Eyes"), exist_ok=True)
    os.makedirs(os.path.join(split, "Open_Eyes"), exist_ok=True)

# --- Nhãn Closed_Eyes: lấy TOÀN BỘ ảnh trong Closed/ ---
closed_paths = list_image_paths(SOURCE_CLOSED)
print(f"Nhãn Closed_Eyes — tổng số ảnh (sử dụng toàn bộ): {len(closed_paths)}")

# --- Nhãn Open_Eyes: random.sample, tối đa 100 ảnh / thư mục (5 thư mục) ---
open_paths = []
for sub in OPEN_SUBFOLDERS:
    folder = os.path.join(EYE_GAZE_ROOT, sub)
    imgs = list_image_paths(folder)
    n_avail = len(imgs)
    k = min(SAMPLES_PER_OPEN_FOLDER, n_avail)
    if k < SAMPLES_PER_OPEN_FOLDER:
        print(
            f"[Cảnh báo] {sub}: chỉ có {n_avail} ảnh (yêu cầu {SAMPLES_PER_OPEN_FOLDER}). "
            f"Lấy tối đa {k} ảnh."
        )
    chunk = random.sample(imgs, k) if k else []
    open_paths.extend(chunk)
    print(f"Open_Eyes — {sub}: đã lấy {len(chunk)} / {n_avail} ảnh")

print(f"Open_Eyes — tổng sau khi gộp năm thư mục: {len(open_paths)}")
print(f"Số lượng so sánh lớp: Closed_Eyes={len(closed_paths)} | Open_Eyes={len(open_paths)}")

# --- Chia 80% train / 20% validation và sao chép ---
print("\nĐang chia tỷ lệ 80% train — 20% validation và ghi vào /kaggle/working/smartdrive_data/ ...")
split_list_and_copy(
    closed_paths,
    os.path.join(TRAIN_DIR, "Closed_Eyes"),
    os.path.join(VAL_DIR, "Closed_Eyes"),
    "ce",
)
split_list_and_copy(
    open_paths,
    os.path.join(TRAIN_DIR, "Open_Eyes"),
    os.path.join(VAL_DIR, "Open_Eyes"),
    "oe",
)


def _count_jpg(p: str) -> int:
    return sum(
        1
        for f in os.listdir(p)
        if os.path.isfile(os.path.join(p, f)) and f.lower().endswith(tuple(IMAGE_EXTS))
    ) if os.path.isdir(p) else 0


print("\n--- Tóm tắt số tệp ảnh theo thư mục đích ---")
print(f"  Train / Closed_Eyes : {_count_jpg(os.path.join(TRAIN_DIR, 'Closed_Eyes'))}")
print(f"  Val   / Closed_Eyes : {_count_jpg(os.path.join(VAL_DIR, 'Closed_Eyes'))}")
print(f"  Train / Open_Eyes   : {_count_jpg(os.path.join(TRAIN_DIR, 'Open_Eyes'))}")
print(f"  Val   / Open_Eyes   : {_count_jpg(os.path.join(VAL_DIR, 'Open_Eyes'))}")
print("Hoàn tất chuẩn bị dữ liệu.")

🚀 Bắt đầu quá trình CHIA MÂM (80% Train - 20% Val)...


Copying Val - Front: 100%|██████████| 124/124 [00:01<00:00, 103.78it/s]


🎉 HOÀN THÀNH BƯỚC 2! Data đã sẵn sàng để đưa vô Lò bát quái!
Số ảnh Train - Nhắm mắt: 401
Số ảnh Train - Mở mắt: 492


## Kiểm tra thư mục đầu ra và đếm ảnh theo nhãn

**Mục đích:** Xác nhận cấu trúc thư mục sau khi sao chép; đối chiếu số lượng ảnh theo từng nhãn trên tập train và validation.

In [3]:
import os

BASE_DIR = "/kaggle/working/smartdrive_data"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "val")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def count_images(folder: str) -> int:
    if not os.path.isdir(folder):
        return 0
    n = 0
    for f in os.listdir(folder):
        p = os.path.join(folder, f)
        if os.path.isfile(p) and os.path.splitext(f)[1].lower() in IMAGE_EXTS:
            n += 1
    return n


print("Thư mục đầu ra: /kaggle/working/smartdrive_data/")
if os.path.isdir(BASE_DIR):
    print("  Nội dung:", ", ".join(os.listdir(BASE_DIR)))
else:
    print("  (Chưa tồn tại — chạy cell chuẩn bị dữ liệu trước.)")

print("\nSố lượng ảnh theo nhãn (đối chiếu train / validation):")
for split_name, root in [("train", TRAIN_DIR), ("val", VAL_DIR)]:
    print(f"\n  [{split_name}]")
    for cls in ("Closed_Eyes", "Open_Eyes"):
        p = os.path.join(root, cls)
        print(f"    {cls}: {count_images(p)}")

if os.path.isdir(TRAIN_DIR):
    tr_c = count_images(os.path.join(TRAIN_DIR, "Closed_Eyes"))
    tr_o = count_images(os.path.join(TRAIN_DIR, "Open_Eyes"))
    va_c = count_images(os.path.join(VAL_DIR, "Closed_Eyes"))
    va_o = count_images(os.path.join(VAL_DIR, "Open_Eyes"))
    print("\nTổng số mẫu — Train:", tr_c + tr_o, "| Validation:", va_c + va_o)

📂 Đang soi đèn pin vô xưởng /kaggle/working/smartdrive_data/:
['val', 'train']

📦 Trong mâm Train có gì?:
['Open_Eyes', 'Closed_Eyes']

📦 Trong mâm Val có gì?:
['Open_Eyes', 'Closed_Eyes']


## Thống kê dataset SDDD gốc (tuỳ chọn)

**Mục đích:** Quét đệ quy đường dẫn dataset đầu vào trên Kaggle và báo cáo tổng số ảnh, video và phân bổ theo thư mục con (hữu ích khi mô tả dữ liệu trong luận văn).

In [4]:
import os

# Đường dẫn gốc dataset SDDD trên Kaggle (kiểm tra đúng đường dẫn input khi chạy)
DATASET_PATH = '/kaggle/input/datasets/rahimulmazumdar/simulated-driver-distraction-dataset-sddd/Dataset'

total_images = 0
total_videos = 0
folder_stats = {}

image_exts = {'.jpg', '.jpeg', '.png'}
video_exts = {'.mp4', '.avi', '.mov'}

print("Đang quét thư mục dataset và thống kê số lượng tệp...\n")
print("-" * 50)

for root, dirs, files in os.walk(DATASET_PATH):
    img_count = 0
    vid_count = 0
    
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in image_exts:
            img_count += 1
            total_images += 1
        elif ext in video_exts:
            vid_count += 1
            total_videos += 1
            
    if img_count > 0 or vid_count > 0:
        folder_name = os.path.relpath(root, DATASET_PATH)
        folder_stats[folder_name] = {'images': img_count, 'videos': vid_count}

print("Tổng hợp dataset SDDD:")
print(f"  Tổng số ảnh  : {total_images:,}")
print(f"  Tổng số video: {total_videos:,}")
print("-" * 50)

print("Chi tiết theo thư mục:")
for folder, stats in sorted(folder_stats.items()):
    print(f"  {folder:<38} | ảnh: {stats['images']:>5} | video: {stats['videos']:>3}")

print("-" * 50)
print("Hoàn tất thống kê.")

🕵️‍♀️ AI Lead đang cử trinh sát đi đếm tài sản... Đợi xíu nha!

--------------------------------------------------
📊 TỔNG KẾT KHO BÁU SDDD 2.84GB:
🖼️ Tổng số hình ảnh: 6,159 tấm
🎬 Tổng số video:    65 clip
--------------------------------------------------
📁 CHI TIẾT TỪNG NGÓC NGÁCH:
 📂 Image Data/Eye Gaze/Closed          |   502 ảnh |   0 video
 📂 Image Data/Eye Gaze/Down            |   481 ảnh |   0 video
 📂 Image Data/Eye Gaze/Front           |   616 ảnh |   0 video
 📂 Image Data/Eye Gaze/Left            |   333 ảnh |   0 video
 📂 Image Data/Eye Gaze/Right           |   553 ảnh |   0 video
 📂 Image Data/Eye Gaze/Up              |   520 ảnh |   0 video
 📂 Image Data/Head Pose/Diagonal Down Left |   241 ảnh |   0 video
 📂 Image Data/Head Pose/Diagonal Down Right |   220 ảnh |   0 video
 📂 Image Data/Head Pose/Diagonal Up Left |   232 ảnh |   0 video
 📂 Image Data/Head Pose/Diagonal Up Right |   277 ảnh |   0 video
 📂 Image Data/Head Pose/Down           |   252 ảnh |   0 video
 📂 Image

## Siêu tham số, tăng cường dữ liệu và luồng ảnh (generator)

**Mục đích:** Đặt kích thước ảnh, batch, epoch và tốc độ học; tạo **`tf.data.Dataset`** bằng `image_dataset_from_directory` (train xáo trộn, validation không xáo trộn). Chuẩn hóa và tăng cường (xoay, zoom, lật) được đặt trong **mô hình** ở cell kế tiếp để tránh lỗi tương thích Keras 3 với `ImageDataGenerator`.

In [5]:
import tensorflow as tf

# --- Siêu tham số ---
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001
DATA_SEED = 42

TRAIN_DIR = '/kaggle/working/smartdrive_data/train'
VAL_DIR = '/kaggle/working/smartdrive_data/val'

AUTOTUNE = tf.data.AUTOTUNE

print(f"Cấu hình huấn luyện: số epoch = {EPOCHS}, kích thước batch = {BATCH_SIZE}.")

# --- Luồng tf.data (tương thích Keras 3; tránh lỗi với ImageDataGenerator + model.fit) ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='binary',
    shuffle=True,
    seed=DATA_SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='binary',
    shuffle=False,
)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

n_train = train_ds.cardinality()
n_train = int(n_train.numpy()) if n_train.numpy() >= 0 else None
steps_msg = (
    f"{n_train} batch/epoch"
    if n_train is not None
    else "số batch do Keras suy ra từ tập dữ liệu"
)
print("\nĐã tạo tf.data.Dataset từ thư mục train và validation (ảnh xám, nhãn nhị phân).")
print(f"Số batch mỗi epoch (ước lượng): {steps_msg}.")
print(
    "Tăng cường dữ liệu (xoay, zoom, lật) và chuẩn hóa [0,1] được đặt trong mô hình ở cell tiếp theo; "
    "validation chạy với training=False nên không áp dụng tăng cường."
)

⏳ Bảng điều khiển: Đã lên lịch trình luyện đan trong 30 vòng!

🔥 Đang cắm ống hút data từ ổ cứng vào Lò...
Found 893 images belonging to 2 classes.
Found 225 images belonging to 2 classes.

📏 Phân tích Lịch trình: Để xong 1 vòng (Epoch), con GPU P100 sẽ phải nhai 27 mẻ ảnh.
💡 Yên tâm: Chút nữa vô Cell 3, tui sẽ cài cái 'Đồng hồ đếm ngược' cho bà xem tận mắt luôn!


## Kiến trúc CNN và biên dịch mô hình

**Mục đích:** Xây dựng mạng tích chập—gộp cực đại—fully connected với dropout; thêm **Rescaling** và **tăng cường ảnh** (xoay, zoom, lật) sau `Input` — chỉ hoạt động khi `training=True`, nên validation không bị làm biến dạng ảnh; đầu ra sigmoid; biên dịch với Adam và `binary_crossentropy`.

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    RandomFlip,
    RandomRotation,
    RandomZoom,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.optimizers import Adam

print("Đang khởi tạo mô hình CNN (Sequential, tf.data + Keras 3)...\n")

# Rescaling + augmentation ở đầu mô hình: train dùng training=True, validation/predict dùng training=False → không tăng cường trên val.
model = Sequential([
    Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 1)),
    Rescaling(1.0 / 255.0),
    RandomRotation(15.0 / 360.0),
    RandomZoom(height_factor=0.1, width_factor=0.1),
    RandomFlip("horizontal"),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

print("Khởi tạo và biên dịch xong. Tóm tắt kiến trúc:")
model.summary()

🧠 Đang xây lại nơ-ron thần kinh (Bản vá lỗi Keras 3)...



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1773995150.981795      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


✅ Xây não bản vá lỗi xong! Xem lại bản thiết kế đi bệ hạ:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 682,753 (2.60 MB)

 Trainable params: 682,753 (2.60 MB)

 Non-trainable params: 0 (0.00 B)

## Huấn luyện: callback lưu checkpoint, dừng sớm và ước lượng thời gian

**Mục đích:** Đăng ký `ModelCheckpoint` (lưu mô hình tốt nhất theo `val_accuracy`), `EarlyStopping` (giảm quá khớp / tiết kiệm thời gian), và callback tùy chỉnh để in thời gian mỗi epoch và thời gian còn lại dự kiến; gọi `model.fit`.

In [7]:
import time
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback

print("Đang thiết lập callback: checkpoint, early stopping, ước lượng thời gian.\n")


class TimeEstimator(Callback):
    """In thời gian mỗi epoch và thời gian còn lại dự kiến (theo trung bình đến hiện tại)."""

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        self.total_epochs = self.params['epochs']

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.time() - self.epoch_start_time
        elapsed_time = time.time() - self.start_time
        avg_epoch_time = elapsed_time / (epoch + 1)
        remaining_epochs = self.total_epochs - (epoch + 1)
        est_remaining_time = avg_epoch_time * remaining_epochs

        print(
            f"  Epoch {epoch + 1}: thời gian epoch = {epoch_time:.0f} s | "
            f"thời gian còn lại ước lượng ≈ {est_remaining_time / 60:.1f} phút"
        )


# Định dạng .h5: phù hợp pipeline hiện tại (vd. realtime_test.py). Keras 3 có thể cảnh báo legacy nhưng vẫn lưu/tải được.
CHECKPOINT_PATH = 'smartdrive_eye_model.h5'

checkpoint = ModelCheckpoint(
    CHECKPOINT_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1,
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

my_callbacks = [checkpoint, early_stop, TimeEstimator()]

print("Bắt đầu huấn luyện (model.fit) với tf.data.Dataset...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=my_callbacks,
)

print(
    f"\nHuấn luyện kết thúc. Checkpoint tốt nhất: `{CHECKPOINT_PATH}` "
    "(ví dụ thư mục `/kaggle/working/` trên Kaggle)."
)

⚙️ Đang setup hệ thống Phao cứu sinh và Đồng hồ đếm ngược...

🚀 LÊN MÂM!!! BẮT ĐẦU ÉP XUNG GPU P100...
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1773995157.229222     148 service.cc:152] XLA service 0x7a1ba0004c50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773995157.229259     148 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773995157.622188     148 cuda_dnn.cc:529] Loaded cuDNN version 91002


 8/28 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5719 - loss: 0.6836

I0000 00:00:1773995160.187800     148 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.6716 - loss: 0.5974 
Epoch 1: val_accuracy improved from -inf to 0.91111, saving model to smartdrive_eye_model.h5


...

 ⏱️ Vòng 2 mất 1s | Dự kiến xong hết trong: 2.1 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9304 - loss: 0.1861 - val_accuracy: 0.9644 - val_loss: 0.0952
Epoch 3/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9577 - loss: 0.1483
Epoch 3: val_accuracy did not improve from 0.96444
 ⏱️ Vòng 3 mất 1s | Dự kiến xong hết trong: 1.5 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9580 - loss: 0.1475 - val_accuracy: 0.9644 - val_loss: 0.1039
Epoch 4/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9747 - loss: 0.0851
Epoch 4: val_accuracy did not improve from 0.96444
 ⏱️ Vòng 4 mất 1s | Dự kiến xong hết trong: 1.2 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9743 - loss: 0.0868 - val_accuracy: 0.9644 - val_loss: 0.0943
Epoch 5/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9706 - loss: 0.1098
Epoch 5: val_accuracy improved from 0.96444 to 0.96889, saving model to smartdrive_eye_model.h5


 ⏱️ Vòng 5 mất 1s | Dự kiến xong hết trong: 1.0 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9703 - loss: 0.1106 - val_accuracy: 0.9689 - val_loss: 0.1012
Epoch 6/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9735 - loss: 0.0916
Epoch 6: val_accuracy improved from 0.96889 to 0.97333, saving model to smartdrive_eye_model.h5


 ⏱️ Vòng 6 mất 1s | Dự kiến xong hết trong: 0.9 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9732 - loss: 0.0926 - val_accuracy: 0.9733 - val_loss: 0.0686
Epoch 7/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9700 - loss: 0.1037
Epoch 7: val_accuracy did not improve from 0.97333
 ⏱️ Vòng 7 mất 1s | Dự kiến xong hết trong: 0.8 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9704 - loss: 0.1036 - val_accuracy: 0.9644 - val_loss: 0.0911
Epoch 8/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9816 - loss: 0.0679
Epoch 8: val_accuracy did not improve from 0.97333
 ⏱️ Vòng 8 mất 1s | Dự kiến xong hết trong: 0.7 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9814 - loss: 0.0684 - val_accuracy: 0.9689 - val_loss: 0.0574
Epoch 9/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9795 - loss: 0.0706
Epoch 9: val_accuracy did not improve from 0.97333
 ⏱️ Vòng 9 mất 1s | Dự kiến xong hết trong: 0.6 phút nữa
28/28 ━━━━━━━━━━

 ⏱️ Vòng 10 mất 1s | Dự kiến xong hết trong: 0.6 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9851 - loss: 0.0575 - val_accuracy: 0.9822 - val_loss: 0.0457
Epoch 11/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9833 - loss: 0.0557
Epoch 11: val_accuracy did not improve from 0.98222
 ⏱️ Vòng 11 mất 1s | Dự kiến xong hết trong: 0.5 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9831 - loss: 0.0562 - val_accuracy: 0.9822 - val_loss: 0.0474
Epoch 12/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9793 - loss: 0.0647
Epoch 12: val_accuracy did not improve from 0.98222
 ⏱️ Vòng 12 mất 1s | Dự kiến xong hết trong: 0.5 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9796 - loss: 0.0645 - val_accuracy: 0.9822 - val_loss: 0.0495
Epoch 13/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9643 - loss: 0.1127
Epoch 13: val_accuracy did not improve from 0.98222
 ⏱️ Vòng 13 mất 1s | Dự kiến xong hết trong: 0.4 phút nữa
28/28 

 ⏱️ Vòng 15 mất 1s | Dự kiến xong hết trong: 0.4 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.9933 - loss: 0.0290 - val_accuracy: 0.9867 - val_loss: 0.0283
Epoch 16/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9868 - loss: 0.0474
Epoch 16: val_accuracy did not improve from 0.98667
 ⏱️ Vòng 16 mất 1s | Dự kiến xong hết trong: 0.3 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9868 - loss: 0.0474 - val_accuracy: 0.9867 - val_loss: 0.0279
Epoch 17/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.9933 - loss: 0.0228
Epoch 17: val_accuracy did not improve from 0.98667
 ⏱️ Vòng 17 mất 1s | Dự kiến xong hết trong: 0.3 phút nữa
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9932 - loss: 0.0238 - val_accuracy: 0.9822 - val_loss: 0.0552
Epoch 18/30
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9818 - loss: 0.0768
Epoch 18: val_accuracy did not improve from 0.98667
 ⏱️ Vòng 18 mất 1s | Dự kiến xong hết trong: 0.3 phút nữa
28/28 

## Đánh giá trên tập validation

**Mục đích:** Sau khi `model.fit` hoàn tất, đánh giá trên **tập validation** (`VAL_DIR`): generator với **`shuffle=False`** để thứ tự batch khớp nhãn khi gọi `model.predict`.

**Đầu ra:** độ chính xác, ma trận nhầm lẫn (TP/TN/FP/FN), báo cáo Precision/Recall/F1, ROC-AUC và đường cong ROC, kích thước file `smartdrive_eye_model.h5` (hoặc `.keras` nếu bạn đổi định dạng lưu).

In [ ]:
# =============================================================================
# Đánh giá mô hình — tập validation, shuffle=False, tf.data (khớp Keras 3)
# =============================================================================
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)


def make_val_eval_ds():
    return tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        color_mode="grayscale",
        label_mode="binary",
        shuffle=False,
    ).prefetch(tf.data.AUTOTUNE)


# --- 1) Nhãn thực tế (cùng thứ tự với predict khi shuffle=False) ---
val_eval = make_val_eval_ds()
y_true = np.concatenate([y.numpy() for _, y in val_eval], axis=0).astype(np.int32)
n_val = len(y_true)

class_names = sorted(
    d for d in os.listdir(VAL_DIR) if os.path.isdir(os.path.join(VAL_DIR, d))
)
inv_idx = {i: class_names[i] for i in range(len(class_names))}
label_display = [
    f"Nhắm mắt (class 0 — {inv_idx[0]})",
    f"Mở mắt (class 1 — {inv_idx[1]})",
]

# --- 2) Xác suất dự đoán lớp dương (sigmoid) ---
val_eval = make_val_eval_ds()
y_score = model.predict(val_eval, verbose=1)
y_score = np.asarray(y_score).reshape(-1)[:n_val]

y_pred = (y_score >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
# Quy ước nhị phân với lớp 1 = "Mở mắt" (positive): TP/TN/FP/FN
TN, FP, FN, TP = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]

# --- 3) In báo cáo dạng khối văn (copy được vào slide / báo cáo) ---
sep = "=" * 72
print(f"\n{sep}")
print("  BÁO CÁO ĐÁNH GIÁ — PHÂN LOẠI NHỊ PHÂN (NHẮM MẮT vs MỞ MẮT)")
print(f"{sep}")
print(f"  Tập đánh giá        : Validation ({VAL_DIR})")
print(f"  Số mẫu              : {n_val}")
print(f"  Ngưỡng phân loại    : 0.5 (sigmoid)")
print(f"  Accuracy (tổng)     : {acc:.4f} ({acc * 100:.2f}%)")
print()
print("  —— Ma trận nhầm lẫn (hàng = thực tế, cột = dự đoán) ——")
print(f"                 Dự đoán: Nhắm (0)    Dự đoán: Mở (1)")
print(f"  Thực: Nhắm (0)        {TN:>6} (TN)        {FP:>6} (FP)")
print(f"  Thực: Mở (1)          {FN:>6} (FN)        {TP:>6} (TP)")
print()
print("  Diễn giải (lớp dương = Mở mắt):")
print(f"    TN = thực Nhắm & dự đoán Nhắm  |  TP = thực Mở & dự đoán Mở")
print(f"    FP = thực Nhắm nhưng báo Mở    |  FN = thực Mở nhưng báo Nhắm")
print(f"{sep}\n")

print("  CLASSIFICATION REPORT (Precision / Recall / F1 theo lớp)")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_display,
        digits=4,
        zero_division=0,
    )
)

try:
    auc = roc_auc_score(y_true, y_score)
except ValueError as e:
    auc = float("nan")
    print(f"  [ROC-AUC] Không tính được (cần cả hai lớp trong val): {e}")
else:
    print(f"  ROC-AUC Score       : {auc:.4f}")

# --- 4) Trực quan hóa: Confusion Matrix + ROC ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.set_theme(style="whitegrid")
ax_cm = axes[0]
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    square=True,
    cbar_kws={"shrink": 0.8},
    xticklabels=["Dự đoán: Nhắm (0)", "Dự đoán: Mở (1)"],
    yticklabels=["Thực: Nhắm (0)", "Thực: Mở (1)"],
    ax=ax_cm,
)
ax_cm.set_title("Confusion Matrix (Validation)")
ax_cm.set_ylabel("Thực tế")
ax_cm.set_xlabel("Dự đoán")

ax_roc = axes[1]
if not np.isnan(auc):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ax_roc.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1, label="Ngẫu nhiên (AUC = 0.5)")
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curve (lớp dương = Mở mắt)")
    ax_roc.legend(loc="lower right")
else:
    ax_roc.text(0.5, 0.5, "Không vẽ ROC\n(thiếu lớp trong tập val)", ha="center", va="center")

plt.tight_layout()
plt.show()

# --- 5) Kích thước file checkpoint (ưu tiên .h5 — khớp load_model trong code edge) ---
MODEL_FILES = ("smartdrive_eye_model.h5", "smartdrive_eye_model.keras")
paths_try = []
for name in MODEL_FILES:
    paths_try.extend([name, os.path.join("/kaggle/working", name)])
model_path = next((p for p in paths_try if os.path.isfile(p)), None)
print(f"{sep}")
print("  THÔNG TIN FILE MÔ HÌNH")
print(f"{sep}")
if model_path:
    size_mb = os.path.getsize(model_path) / (1024 * 1024)
    print(f"  Đường dẫn           : {os.path.abspath(model_path)}")
    print(f"  Dung lượng          : {size_mb:.3f} MB")
else:
    print(f"  Không tìm thấy      : {MODEL_FILES} (chạy training trước)")
print(f"{sep}\n")